# Session 1: NumPy Fundamentals (2 hours)

**Goal:** Build the mental model: *read an ML-style task → map it to array shapes and operations → implement in NumPy.* We start with real mini-tasks (cases), not a long list of basics.

| Section | Time | Focus |
|---------|------|-------|
| Setup | 10 min | Import & verify environment |
| Anchor case 1 | 40 min | Weighted sum & batch prediction |
| Anchor case 2 | 30 min | Normalise a vector to 0–1 |
| Anchor case 3 | 35 min | Boolean indexing (extract with mask) |
| Anchor case 4 | 35 min | View vs copy — avoid the bug |
| Reference | 50 min | NumPy essentials drill (if you need more practice) |

## Setup (10 min)

In [1]:
import numpy as np

print(f"NumPy version: {np.__version__}")
print("Setup complete!")

NumPy version: 2.0.2
Setup complete!


---
## Anchor case 1: Weighted sum and batch prediction [EASY]

In ML we often compute **weighted sums**: one number per example (e.g. a score or prediction). Here we do it for **one** example, then for a **batch** of examples (matrix × vector).

### Concept: Arrays and shape

An **array** is a grid of numbers. **Shape** is the size along each dimension: e.g. `(5,)` = 1D vector of length 5, `(3, 4)` = 3 rows, 4 columns. Use `arr.shape` to inspect. Create from a list with `np.array([...])`.

### Worked example: One neuron, then a batch

**Task:** Given scores (features) and weights, compute the weighted sum (like one neuron). Then do it for 5 examples at once: `batch` has shape (5, 2), `weights` (2,). We want one number per example → shape (5,).

**Expert reasoning:** For one example, `scores @ weights` or `np.dot(scores, weights)` gives a scalar. For a batch, we need matrix × vector: (5, 2) @ (2,) → (5,). NumPy's `@` does exactly that. We keep `weights` as (2,) so the last dimension of `batch` (2) matches; the result loses that dimension → (5,).

In [2]:
# One example: 2 features, 2 weights
scores = np.array([0.5, 1.5])   # shape (2,)
weights = np.array([2.0, -0.5]) # shape (2,)
one_pred = np.dot(scores, weights)  # or scores @ weights → scalar
print(f"One prediction: {one_pred}, shape {np.array(one_pred).shape}")

# Batch of 5 examples, 2 features each
batch = np.array([[0.5, 1.5], [1.0, 0.0], [0.0, 1.0], [2.0, 2.0], [0.1, 0.9]])  # (5, 2)
batch_pred = batch @ weights  # (5, 2) @ (2,) → (5,)
print(f"Batch predictions: {batch_pred}")
print(f"Shapes: batch {batch.shape}, weights {weights.shape}, result {batch_pred.shape}")

One prediction: 0.25, shape ()
Batch predictions: [ 0.25  2.   -0.5   3.   -0.25]
Shapes: batch (5, 2), weights (2,), result (5,)


### Your turn [EASY]

Use the same idea: 10 examples, **3** features. `batch_10` has shape (10, 3), `w` has shape (3,). Compute the vector of 10 predictions (shape (10,)).

In [3]:
np.random.seed(42)
batch_10 = np.random.randn(10, 3)  # 10 examples, 3 features
w = np.array([1.0, -0.5, 0.0])

# YOUR CODE HERE: compute predictions, shape (10,)
preds = batch_10 @ w
print(f"Batch predictions: {preds}")

Batch predictions: [ 0.5658463   1.64010654  1.19549545  0.77426889  1.19860239 -0.05587197
 -0.20187222 -0.2595404  -0.59984402  0.67601736]


### Sensemaking

**What would break if `weights` had shape (3,) and `batch` had shape (5, 2)?** Write one sentence (e.g. dimension mismatch, which dimensions?).

_Your answer:_ Dimension mismatch between shape (5, 2) with last dim of 2 and weights (3,) with last dim of 3.

---
## Anchor case 2: Normalise a vector to 0–1 [EASY]

**Task:** Given a vector of numbers, rescale so the minimum becomes 0 and the maximum becomes 1. Formula: `(x - min) / (max - min)`.

### Worked example

In [4]:
v = np.array([10.0, 20.0, 5.0, 35.0, 15.0])
v_min = v.min()
v_max = v.max()
v_normalized = (v - v_min) / (v_max - v_min)
print(f"Original: {v}")
print(f"Normalized: {v_normalized}")
print(f"Min/max of normalized: {v_normalized.min():.2f}, {v_normalized.max():.2f}")

Original: [10. 20.  5. 35. 15.]
Normalized: [0.16666667 0.5        0.         1.         0.33333333]
Min/max of normalized: 0.00, 1.00


### Your turn [EASY]

Normalise a **random** vector of length 7 (use `np.random.rand(7)`). Check that the result has min 0 and max 1.

In [5]:
np.random.seed(0)
vec = np.random.rand(7)
# YOUR CODE HERE
v_min = vec.min()
v_max = vec.max()
v_norm = (vec - v_min) / (v_max - v_min)
print(vec)
print(v_norm)


[0.5488135  0.71518937 0.60276338 0.54488318 0.4236548  0.64589411
 0.43758721]
[0.42931    1.         0.6143648  0.41582851 0.         0.76230862
 0.04778991]


### Sensemaking

**When would you use `reshape(-1, 1)` on a 1D vector in an ML context?** (One sentence: what shape are we aiming for and why?)

_Your answer:_ We want to make shape (n, 1) for mat mul.

---
## Anchor case 3: Extract elements with a mask (boolean indexing) [MEDIUM]

**Task:** We have predictions (n,) and a boolean mask `valid` (n,) indicating which examples are valid (e.g. no missing label). Extract only the predictions where `valid` is True. Then compute the mean of **only those** predictions.

**Why it matters:** In ML we often filter out invalid or padded positions before computing a loss or metric.

### Worked example

In [6]:
predictions = np.array([0.1, 0.9, 0.3, 0.7, 0.5])
valid = np.array([True, True, False, True, False])  # 3 valid
pred_valid = predictions[valid]   # boolean indexing → 1D array of length 3
mean_valid = pred_valid.mean()
print(f"predictions: {predictions}")
print(f"valid:       {valid}")
print(f"pred_valid: {pred_valid}")
print(f"mean_valid: {mean_valid}")

predictions: [0.1 0.9 0.3 0.7 0.5]
valid:       [ True  True False  True False]
pred_valid: [0.1 0.9 0.7]
mean_valid: 0.5666666666666667


### Your turn [MEDIUM]

Given `scores` of shape (20,) and `mask` of shape (20,) (boolean), compute the **maximum** score among the positions where `mask` is True.

In [7]:
np.random.seed(42)
scores = np.random.rand(20)
mask = np.random.rand(20) > 0.5  # about half True
# YOUR CODE HERE
scores_valid = scores[mask]
max_valid = scores_valid.max()
print(scores_valid)
print(max_valid)

[0.37454012 0.15599452 0.86617615 0.60111501 0.02058449 0.21233911
 0.18182497 0.18340451 0.43194502]
0.8661761457749352


### Sensemaking

**What is the shape of `predictions[valid]` if `predictions` has shape (100,) and `valid` has 30 True values?**

_Your answer: _ (30,)

---
## Anchor case 4: View vs copy — avoid a bug [MEDIUM]

**Task:** You take a **slice** of an array (e.g. first 3 elements), modify that slice in-place, and expect the original array to be unchanged. Run the code below: does the original change? Then fix it so the original stays unchanged (use `.copy()` when you need an independent array).

### Worked example: the bug

In [8]:
original = np.array([10, 20, 30, 40, 50])
slice_view = original[1:4]   # slice → view (shares memory)
slice_view[0] = 999
print(f"After modifying slice_view: original = {original}")
print("The original changed! Slicing returns a VIEW.")

After modifying slice_view: original = [ 10 999  30  40  50]
The original changed! Slicing returns a VIEW.


### Your turn [MEDIUM]

Create a **copy** of the first 3 elements, modify the copy (e.g. set copy[0] = -1), and verify that `original` is unchanged.

In [9]:
original = np.array([10, 20, 30, 40, 50])
# YOUR CODE: get first 3 elements as a COPY, then set copy[0] = -1
first_three = original[:3].copy()  # hint: use .copy()
first_three[0] = -1
print(f"first_three: {first_three}")
print(f"original:    {original}")
assert original[0] == 10, "Original should be unchanged"

first_three: [-1 20 30]
original:    [10 20 30 40 50]


### Sensemaking

**When would you prefer a view over a copy?** (Think: memory, speed, safety.)

_Your answer: _ Speed and Memory

---
## Reference: NumPy essentials (if you need more drill)

The anchor cases above cover the mental model we need for ML. Below is a **subset** of exercises that support those cases (array creation, indexing, shapes). Full set: [NumPy-100](https://github.com/rougier/numpy-100). For each one, think: *"How would I do this with a for loop? Why is the NumPy version faster?"*

### Group A: Basic Array Operations (Exercises 1-10)

**Q1.** Create a null vector of size 10

In [10]:
# YOUR CODE HERE
v = np.zeros(10)
v

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

**Q2.** Create a null vector of size 10 but the fifth value is 1

In [11]:
# YOUR CODE HERE
v = np.zeros(10)
v[4] = 1
v

array([0., 0., 0., 0., 1., 0., 0., 0., 0., 0.])

**Q3.** Create a vector with values ranging from 10 to 49

In [12]:
# YOUR CODE HERE
v = np.arange(10, 50)
v


array([10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26,
       27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43,
       44, 45, 46, 47, 48, 49])

**Q4.** Reverse a vector (first element becomes last)

In [13]:
# YOUR CODE HERE
v = np.arange(1,10)
v = v[::-1]
# v = np.flip(v)
v

array([9, 8, 7, 6, 5, 4, 3, 2, 1])

**Q5.** Create a 3x3 matrix with values ranging from 0 to 8

In [14]:
# YOUR CODE HERE
mat = np.arange(9)
mat = mat.reshape(3, 3)
mat

array([[0, 1, 2],
       [3, 4, 5],
       [6, 7, 8]])

**Q6.** Find indices of non-zero elements from `[1, 2, 0, 0, 4, 0]`

In [15]:
# YOUR CODE HERE
v = np.array([1, 2, 0, 0, 4, 0])
zeros = v != 0
indicies = np.arange(len(v))
z_i = indicies[zeros]
z_i

array([0, 1, 4])

**Q7.** Create a 3x3 identity matrix

In [16]:
# YOUR CODE HERE
iden = np.identity(3)
iden

array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]])

### Group B: Indexing and Slicing (Exercises 15-25)

**Q8.** Create a 2D array with 1 on the border and 0 inside (5x5)

In [17]:
# YOUR CODE HERE
mat = np.zeros(25).reshape(5,5)
mat[:, [0, -1]] = 1
mat[[0, -1], :] = 1
mat

array([[1., 1., 1., 1., 1.],
       [1., 0., 0., 0., 1.],
       [1., 0., 0., 0., 1.],
       [1., 0., 0., 0., 1.],
       [1., 1., 1., 1., 1.]])

**Q9.** Pad an existing array (3x3 of ones) with a border of zeros

In [18]:
# YOUR CODE HERE
mat = np.ones(9).reshape(3,3)
mat = np.pad(mat, (1,1))
mat

array([[0., 0., 0., 0., 0.],
       [0., 1., 1., 1., 0.],
       [0., 1., 1., 1., 0.],
       [0., 1., 1., 1., 0.],
       [0., 0., 0., 0., 0.]])

**Q10.** Create a 5x5 matrix with row values ranging from 0 to 4

In [19]:
# YOUR CODE HERE
mat = np.arange(0, 5).reshape(1,-1).repeat(5, 0)
mat

array([[0, 1, 2, 3, 4],
       [0, 1, 2, 3, 4],
       [0, 1, 2, 3, 4],
       [0, 1, 2, 3, 4],
       [0, 1, 2, 3, 4]])

**Q11.** Create a checkerboard 8x8 matrix using slicing

In [20]:
# YOUR CODE HERE
mat = np.zeros(64).reshape(8,8)
mat[0:9:2, 1:9:2] = 1
mat[1:9:2, 0:9:2] = 1
mat


array([[0., 1., 0., 1., 0., 1., 0., 1.],
       [1., 0., 1., 0., 1., 0., 1., 0.],
       [0., 1., 0., 1., 0., 1., 0., 1.],
       [1., 0., 1., 0., 1., 0., 1., 0.],
       [0., 1., 0., 1., 0., 1., 0., 1.],
       [1., 0., 1., 0., 1., 0., 1., 0.],
       [0., 1., 0., 1., 0., 1., 0., 1.],
       [1., 0., 1., 0., 1., 0., 1., 0.]])

**Q12.** Normalize a 5x5 random matrix (values between 0 and 1)

In [21]:
# YOUR CODE HERE
mat = np.random.randint(0, 10, (5,5))
mat_min = mat.min()
mat_max = mat.max()
mat = (mat - mat_min) / (mat_max - mat_min)
mat

array([[0.66666667, 0.77777778, 0.22222222, 0.        , 0.33333333],
       [0.11111111, 0.77777778, 0.33333333, 0.11111111, 0.55555556],
       [0.55555556, 1.        , 0.33333333, 0.55555556, 0.11111111],
       [1.        , 0.11111111, 1.        , 0.33333333, 0.77777778],
       [0.66666667, 0.88888889, 0.77777778, 0.44444444, 0.11111111]])

### Group C: Array Manipulation (Exercises 30-40)

**Q13.** How to find common values between two arrays?

In [22]:
A = np.array([1, 2, 3, 4, 5])
B = np.array([3, 4, 5, 6, 7])

# YOUR CODE HERE
np.intersect1d(A, B)

array([3, 4, 5])

**Q14.** Create a random vector of size 10 and sort it

In [23]:
# YOUR CODE HERE
mat = np.random.randint(0, 10, 10)
mat.sort()
mat

array([0, 4, 6, 7, 7, 8, 8, 8, 8, 9])

**Q15.** Replace the maximum value by 0 in a random vector

In [24]:
# YOUR CODE HERE
mat = np.random.randint(0, 10, 10)
mat_max = mat.max()
mat[mat == mat_max] = 0
mat

array([0, 0, 0, 2, 0, 0, 2, 2, 0, 4])

**Q16.** Subtract the mean of each row of a matrix (5x5 random)

In [25]:
# YOUR CODE HERE
mat = np.random.randint(0, 10, (5,5))
mat_mean = mat.mean(-1, keepdims=True)
print(mat)
print(mat_mean)
print(mat - mat_mean)

[[9 6 9 8 6]
 [8 7 1 0 6]
 [6 7 4 2 7]
 [5 2 0 2 4]
 [2 0 4 9 6]]
[[7.6]
 [4.4]
 [5.2]
 [2.6]
 [4.2]]
[[ 1.4 -1.6  1.4  0.4 -1.6]
 [ 3.6  2.6 -3.4 -4.4  1.6]
 [ 0.8  1.8 -1.2 -3.2  1.8]
 [ 2.4 -0.6 -2.6 -0.6  1.4]
 [-2.2 -4.2 -0.2  4.8  1.8]]


**Q17.** Sort an array by the nth column

In [26]:
# Create a random 5x3 matrix and sort by column index 1
# YOUR CODE HERE
mat = np.random.randint(0, 10, (5,3))
print(mat)
new_i = mat[:, 1].argsort()
mat = mat[new_i]
print(mat)

[[6 8 9]
 [9 2 6]
 [0 3 3]
 [4 6 6]
 [3 6 2]]
[[9 2 6]
 [0 3 3]
 [4 6 6]
 [3 6 2]
 [6 8 9]]


### Group D: Mathematical Operations (Exercises 45-55)

**Q18.** Compute `((A + B) * (-A / 2))` element-wise without creating intermediate arrays (use `np.add`, `np.multiply`, etc. with `out` parameter)

In [80]:
A = np.ones(3) * 1
B = np.ones(3) * 2

# YOUR CODE HERE
np.add(A, B, out=B)
np.divide(A, -2, out=A)
np.multiply(A, B, out=A)
A

array([-1.5, -1.5, -1.5])

**Q19.** Get the n largest values of an array

In [81]:
Z = np.arange(10000)
np.random.shuffle(Z)
n = 5

# YOUR CODE HERE (hint: np.argpartition)
print(Z)
Z[np.argpartition(Z, -n)[-n:]]

[8939 1489  102 ... 5149 8194 2953]


array([9995, 9996, 9997, 9998, 9999])

**Q20.** Compute the sum of a vector using different methods and compare speed

In [29]:
Z = np.random.random(1000000)

# Method 1: Python sum()
%timeit sum(Z)

# Method 2: np.sum()
%timeit np.sum(Z)

# Method 3: Z.sum()
%timeit Z.sum()

# Method 4: np.add.reduce()
%timeit np.add.reduce(Z)

113 ms ± 17.3 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
274 μs ± 25.2 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
248 μs ± 3.33 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
274 μs ± 17.4 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


**Q21.** Compute the dot product of two vectors manually (without np.dot), then verify

In [49]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

# Manual dot product
dot_manual = a[0] * b[0] + a[1] * b[1] + a[2] * b[2]

# Verify with NumPy
dot_numpy = np.dot(a, b)

print(f"Manual: {dot_manual}, NumPy: {dot_numpy}, Match: {dot_manual == dot_numpy}")

Manual: 32, NumPy: 32, Match: True


**Q22.** Compute running (cumulative) sum of a vector

In [50]:
Z = np.array([1, 2, 3, 4, 5])

# YOUR CODE HERE
np.cumsum(Z)


array([ 1,  3,  6, 10, 15])

### Group E: Linear Algebra Basics (Exercises 60-65)

**Q23.** Multiply a 5x3 matrix by a 3x2 matrix (real matrix product)

In [ ]:
# YOUR CODE HERE
A = np.arange(15).reshape(5,3)
B = np.arange(6).reshape(3,2)

# print(A)
# print(B)
np.matmul(A, B)

[[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]
 [12 13 14]]
[[0 1]
 [2 3]
 [4 5]]


array([[ 10,  13],
       [ 28,  40],
       [ 46,  67],
       [ 64,  94],
       [ 82, 121]])

**Q24.** Create a structured array with `name` (string) and `age` (int) fields

In [83]:
# YOUR CODE HERE
dtype = [ ('name', 'U10'), ('age', "i4")]
arr = np.array([], dtype=dtype)

**Q25.** Given a 1D array, negate all elements between 3 and 8 (in-place)

In [85]:
Z = np.arange(11)
# print(Z)
# YOUR CODE HERE
indices = (3 <= Z) & (Z <= 8)
# print(indices)
Z[indices] *= -1
print(Z)

[ 0  1  2 -3 -4 -5 -6 -7 -8  9 10]


---
## Session 1 Reflection

**What concept was most surprising?**

Partitions, Argsort, etc.

**What still feels fuzzy?**

Mostly remembering the name of the functions

**Connection:** In Session 3 we'll use the same "batch @ weights" idea for linear regression predictions. In Session 5 we assemble everything into one model.